In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv('loan_approval_data.csv')

print("Columns:", df.columns.tolist())
print("\nShape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Types & Missing Values:")
print(df.info())

Columns: ['Applicant_ID', 'Applicant_Income', 'Coapplicant_Income', 'Employment_Status', 'Age', 'Marital_Status', 'Dependents', 'Credit_Score', 'Existing_Loans', 'DTI_Ratio', 'Savings', 'Collateral_Value', 'Loan_Amount', 'Loan_Term', 'Loan_Purpose', 'Property_Area', 'Education_Level', 'Gender', 'Employer_Category', 'Loan_Approved']

Shape: (1000, 20)

First few rows:
   Applicant_ID  Applicant_Income  Coapplicant_Income Employment_Status   Age  \
0           1.0           17795.0              1387.0          Salaried  51.0   
1           2.0            2860.0              2679.0          Salaried  46.0   
2           3.0            7390.0              2106.0          Salaried  25.0   
3           4.0           13964.0              8173.0          Salaried  40.0   
4           5.0           13284.0              4223.0     Self-employed  31.0   

  Marital_Status  Dependents  Credit_Score  Existing_Loans  DTI_Ratio  \
0        Married         0.0         637.0             4.0       0.53 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load dataset
df = pd.read_csv('loan_approval_data.csv')

# Drop missing target rows
df = df.dropna(subset=['Loan_Approved'])

# Separate features and target
X = df.drop(columns=['Applicant_ID', 'Loan_Approved'])
y = df['Loan_Approved'].map({'Yes': 1, 'No': 0})

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Define Preprocessing Pipelines
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit Decision Tree Model
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=4, random_state=42))
])

clf.fit(X_train, y_train)

# Predictions & Metrics
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Extract feature names after one-hot encoding
cat_encoder = clf.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
encoded_cat_cols = cat_encoder.get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + encoded_cat_cols

# Feature Importances
tree_model = clf.named_steps['classifier']
importances = tree_model.feature_importances_
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False).head(10)

print("\nTop 10 Feature Importances:")
print(importance_df)

# Plot Decision Tree Visualization
plt.figure(figsize=(20, 10))
plot_tree(tree_model, feature_names=feature_names, class_names=['No', 'Yes'], filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Visualization (Max Depth = 4)", fontsize=16)
plt.savefig('decision_tree.png', bbox_inches='tight')
plt.close()

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.close()

Accuracy: 0.9421
Precision: 0.8769
Recall: 0.9500
F1 Score: 0.9120

Top 10 Feature Importances:
              Feature  Importance
6           DTI_Ratio    0.504685
4        Credit_Score    0.333697
0    Applicant_Income    0.125737
9         Loan_Amount    0.035881
1  Coapplicant_Income    0.000000
2                 Age    0.000000
5      Existing_Loans    0.000000
3          Dependents    0.000000
7             Savings    0.000000
8    Collateral_Value    0.000000


In [3]:
# Print text representation of decision tree rules
tree_text = export_text(tree_model, feature_names=feature_names)
print("Decision Tree Rules:\n")
print(tree_text)

Decision Tree Rules:

|--- Credit_Score <= 650.50
|   |--- class: 0
|--- Credit_Score >  650.50
|   |--- DTI_Ratio <= 0.39
|   |   |--- Applicant_Income <= 5355.50
|   |   |   |--- Loan_Amount <= 11528.00
|   |   |   |   |--- class: 1
|   |   |   |--- Loan_Amount >  11528.00
|   |   |   |   |--- class: 0
|   |   |--- Applicant_Income >  5355.50
|   |   |   |--- Applicant_Income <= 7271.50
|   |   |   |   |--- class: 1
|   |   |   |--- Applicant_Income >  7271.50
|   |   |   |   |--- class: 1
|   |--- DTI_Ratio >  0.39
|   |   |--- class: 0



In [5]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

# 1. Load data
df = pd.read_csv('loan_approval_data.csv').dropna(subset=['Loan_Approved'])

X = df.drop(columns=['Applicant_ID', 'Loan_Approved'])
y = df['Loan_Approved'].map({'Yes': 1, 'No': 0})

# 2. Identify numerical and categorical features
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# 3. Pipelines for preprocessing
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols),
    ]
)

# 4. Model definition
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(max_depth=4, random_state=42)),
    ]
)

# 5. Train model
model.fit(X, y)

# 6. Save model to file
joblib.dump(model, 'model.pkl')
print("Model successfully saved as model.pkl")

Model successfully saved as model.pkl


In [6]:
import joblib
import pandas as pd

# Load saved model
model = joblib.load('model.pkl')

# Predict on new data
# prediction = model.predict(input_dataframe)